# 📊 실습 2 — 공개 데이터로 '논문 속 인과 그래프' 그리기

OpenAI 키 없이, **연구팀이 공개한 실제 엣지 데이터**(2,436개 엣지)로 그래프 4종을 그려봅니다.

1. 관계 유형 분포 (어떤 관계가 많은가)
2. 허브 노드 Top 15 (무엇이 네트워크의 중심인가)
3. 네트워크 백본 (빈도 높은 연결만 모은 전체 지도)
4. 특정 노드의 이웃 (ego network — 예: aggregation)

> 데이터 출처: https://github.com/STARG-LEE/mab-causal-network-v2

In [ ]:
!pip install -q pandas networkx matplotlib koreanize-matplotlib
import pandas as pd, networkx as nx, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
try:
    import koreanize_matplotlib            # 한글 깨짐 방지(자동으로 한글 폰트 적용)
except Exception:
    for _f in ['Malgun Gothic','AppleGothic','NanumGothic']:   # 폴백
        try: plt.rcParams['font.family']=_f; break
        except Exception: pass
plt.rcParams['axes.unicode_minus']=False
%matplotlib inline

URL = 'https://raw.githubusercontent.com/STARG-LEE/mab-causal-network-v2/main/step2c_v4d_causal_edges_summary.csv'
df = pd.read_csv(URL)
print('엣지', len(df), '| 컬럼', list(df.columns))
df.head()

## 색 규칙 정의 (웹 시각화와 동일)
노드=카테고리 6색, 화살표=방향성 3색(초록/빨강/회색).

In [ ]:
CAT_COLOR = {'sequence':'#c4b5fd','structure':'#a78bfa','formulation':'#34d399',
             'stress':'#fbbf24','stability':'#f87171','quality_outcome':'#fb923c'}
CAT_KOR = {'sequence':'Sequence(서열)','structure':'Structure(구조)','formulation':'Formulation(제형)',
           'stress':'Stress(스트레스)','stability':'Stability(안정성)','quality_outcome':'Quality(품질)'}
POSITIVE = {'stabilizes','decreases','inhibits','prevents','shields'}
NEGATIVE = {'destabilizes','increases','promotes','induces','aggregates','oxidizes',
            'deamidates','isomerizes','fragments','unfolds','adsorbs','precipitates','degrades','denatures'}
def rel_color(r):
    if r in POSITIVE: return '#22c55e'
    if r in NEGATIVE: return '#ef4444'
    return '#94a3b8'

node_cat, strength = {}, {}
for _, r in df.iterrows():
    node_cat.setdefault(str(r['cause']),  str(r['category_cause']))
    node_cat.setdefault(str(r['effect']), str(r['category_effect']))
    strength[str(r['cause'])]  = strength.get(str(r['cause']),0)  + int(r['frequency'])
    strength[str(r['effect'])] = strength.get(str(r['effect']),0) + int(r['frequency'])
print('노드 수:', len(node_cat))

## 그림 1 — 관계 유형 분포

In [ ]:
vc = df['relationship'].value_counts().head(15)[::-1]
plt.figure(figsize=(9,6))
plt.barh(vc.index, vc.values, color=[rel_color(r) for r in vc.index], edgecolor='white')
for y,v in enumerate(vc.values): plt.text(v+3,y,f'{v:,}',va='center',fontsize=9)
plt.title('관계 유형 분포 Top 15  (초록=안정화·빨강=불안정·회색=중립)',fontweight='bold')
plt.xlabel('엣지 수'); plt.tight_layout(); plt.show()

💬 **읽기:** `modifies`, `correlates`, `decreases` 가 가장 많습니다. 즉, 문헌은 단정적인 인과보다 '수식/상관' 같은 **중립적·정량적 표현**을 많이 씁니다.

## 그림 2 — 허브 노드 Top 15

In [ ]:
s = pd.Series(strength).sort_values(ascending=False).head(15)[::-1]
plt.figure(figsize=(9,6))
plt.barh(s.index, s.values, color=[CAT_COLOR.get(node_cat.get(n,''),'#888') for n in s.index], edgecolor='white')
for y,v in enumerate(s.values): plt.text(v+5,y,f'{v:,}',va='center',fontsize=9)
plt.title('허브 노드 Top 15  (막대=총 빈도, 색=카테고리)',fontweight='bold')
plt.xlabel('총 빈도(in+out)')
plt.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor=c,markersize=10,label=CAT_KOR[k]) for k,c in CAT_COLOR.items()],
           loc='lower right',fontsize=8)
plt.tight_layout(); plt.show()

💬 **읽기:** `aggregation`(응집), `immunogenicity`(면역원성), `binding activity`(결합능)이 네트워크의 **허브**입니다 — mAb 안정성 문제의 길목.

## 그림 3 — 네트워크 백본 (빈도 상위 120 엣지)

In [ ]:
sub = df.sort_values('frequency', ascending=False).head(120)
G = nx.DiGraph()
for _, r in sub.iterrows():
    G.add_edge(str(r['cause']), str(r['effect']), rel=str(r['relationship']), w=int(r['frequency']))
pos = nx.spring_layout(G, k=0.9, iterations=200, seed=42)
fig,ax = plt.subplots(figsize=(15,11)); fig.patch.set_facecolor('#0a0e17'); ax.set_facecolor('#0a0e17')
nx.draw_networkx_edges(G,pos,edge_color=[rel_color(G[u][v]['rel']) for u,v in G.edges()],
                       width=[0.5+G[u][v]['w']*0.12 for u,v in G.edges()],alpha=0.55,arrows=True,
                       arrowsize=9,connectionstyle='arc3,rad=0.08',ax=ax)
nx.draw_networkx_nodes(G,pos,node_color=[CAT_COLOR.get(node_cat.get(n,''),'#888') for n in G.nodes()],
                       node_size=[120+strength.get(n,0)*4 for n in G.nodes()],edgecolors='#0a0e17',linewidths=0.5,ax=ax)
big = sorted(G.nodes(), key=lambda n:-strength.get(n,0))[:22]
nx.draw_networkx_labels(G,pos,labels={n:n for n in big},font_size=8,font_color='#e8ecf4',ax=ax)
ax.set_title('mAb 안정성 인과 네트워크 — 백본(상위 120 엣지)',color='#e8ecf4',fontsize=15,fontweight='bold')
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor=c,markersize=11,label=CAT_KOR[k]) for k,c in CAT_COLOR.items()],
          loc='upper left',framealpha=0.2,labelcolor='#e8ecf4',fontsize=9)
ax.axis('off'); plt.tight_layout(); plt.show()

## 그림 4 — 특정 노드의 이웃만 보기 (ego network)

`CENTER` 를 바꿔서 다른 노드도 살펴보세요 (예: `viscosity`, `oxidation`, `immunogenicity`).

In [ ]:
CENTER = 'aggregation'   # ← 바꿔보세요
TOP = 18
rel = df[(df['cause']==CENTER)|(df['effect']==CENTER)].sort_values('frequency',ascending=False).head(TOP)
G = nx.DiGraph()
for _, r in rel.iterrows():
    G.add_edge(str(r['cause']), str(r['effect']), rel=str(r['relationship']), w=int(r['frequency']))
pos = nx.spring_layout(G, k=1.2, iterations=200, seed=7)
plt.figure(figsize=(13,9))
nx.draw_networkx_edges(G,pos,edge_color=[rel_color(G[u][v]['rel']) for u,v in G.edges()],
                       width=[1.0+G[u][v]['w']*0.15 for u,v in G.edges()],alpha=0.7,arrows=True,
                       arrowsize=16,connectionstyle='arc3,rad=0.08')
nx.draw_networkx_nodes(G,pos,node_color=[CAT_COLOR.get(node_cat.get(n,''),'#888') for n in G.nodes()],
                       node_size=[2600 if n==CENTER else 900 for n in G.nodes()],edgecolors='#333',linewidths=1.0)
nx.draw_networkx_labels(G,pos,font_size=9)
nx.draw_networkx_edge_labels(G,pos,edge_labels={(u,v):G[u][v]['rel'] for u,v in G.edges()},font_size=7,
                             font_color='#444',bbox=dict(boxstyle='round,pad=0.1',fc='white',ec='none',alpha=0.6))
plt.title(f"'{CENTER}' 노드 중심 인과 이웃 (ego network)",fontsize=14,fontweight='bold')
plt.axis('off'); plt.tight_layout(); plt.show()

## 정리
- 색·크기·화살표 **세 가지 시각 신호**만으로 수천 편 논문의 인과 지식을 한 장에 담았습니다.
- 같은 데이터가 **GATED 모델**의 학습 재료가 됩니다(다음 단계).
- 마우스로 직접 만지는 버전: **https://starg-lee.github.io/mab-causal-network-v2/**